In [ ]:
!pip install torch torchvision opencv-python pyyaml pillow matplotlib


## **📑 Explicação das Camadas**
### **Backbone (VGG16)**
- Rede **mais profunda**, ideal para detecção de objetos.
- **Inicializada sem pesos pré-treinados (`weights=None`).**
- **512 canais na última camada convolucional.**

### **Region Proposal Network (RPN)**
- Gera regiões candidatas para objetos.

### **ROI Pooling**
- Normaliza as regiões propostas para o mesmo tamanho.

### **Cabeça da Rede (Head)**
- Classifica objetos e ajusta bounding boxes.


In [ ]:
import os
import time
import torch
import torchvision
import numpy as np
import matplotlib.pyplot as plt
import cv2
import yaml
import torch.nn as nn
import torch.nn.init as init
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
from torchvision.models import vgg16
from torchvision.models.detection import FasterRCNN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign
from PIL import Image

# Configuração do dispositivo (Apenas CPU)
device = torch.device("cpu")
print(f"🖥️ Usando dispositivo: {device}")

# Carregar configurações do dataset
print("📂 Carregando configurações do dataset...")
with open("dataset.yaml", "r") as file:
    dataset_config = yaml.safe_load(file)

train_dir = dataset_config["train"]
val_dir = dataset_config["val"]
nc = dataset_config["nc"]
class_names = dataset_config["names"]
print("✅ Configurações carregadas com sucesso!")

# Criar Backbone (VGG16 do Zero)
vgg = vgg16(weights=None)  # Sem pesos pré-treinados
backbone = vgg.features  # Pegamos apenas as camadas convolucionais da VGG16
backbone.out_channels = 512  # A última camada convolucional da VGG16 tem 512 canais

# Inicializar pesos aleatórios
def initialize_weights(m):
    if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
        init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

backbone.apply(initialize_weights)
print("✅ Backbone (VGG16) criado e inicializado do zero!")

# Criar Region Proposal Network (RPN)
anchor_generator = AnchorGenerator(
    sizes=((16, 32, 64, 128, 256),),
    aspect_ratios=((0.5, 1.0, 2.0),)
)
print("✅ RPN configurado!")

# Criar ROI Pooling
roi_pooler = MultiScaleRoIAlign(
    featmap_names=["0"],
    output_size=7,
    sampling_ratio=2
)
print("✅ ROI Pooling configurado!")

# Criar modelo Faster R-CNN com VGG16 como Backbone
model = FasterRCNN(
    backbone,
    num_classes=nc + 1,
    rpn_anchor_generator=anchor_generator,
    box_roi_pool=roi_pooler
)
model.to(device)
print("✅ Modelo Faster R-CNN com VGG16 criado do zero!")

# Criar Dataset
class ObjectDetectionDataset(Dataset):
    def __init__(self, image_dir, transform=None):
        self.image_dir = image_dir
        self.transform = transform
        self.image_filenames = sorted([f for f in os.listdir(image_dir) if f.endswith(('.png', '.jpg', '.jpeg'))])
        self.label_dir = image_dir.replace("images", "labels")

    def __len__(self):
        return len(self.image_filenames)

    def __getitem__(self, idx):
        img_path = os.path.join(self.image_dir, self.image_filenames[idx])
        label_path = os.path.join(self.label_dir, self.image_filenames[idx].replace(".png", ".txt").replace(".jpg", ".txt").replace(".jpeg", ".txt"))

        if not os.path.exists(label_path):
            return torch.zeros(3, 224, 224), {"boxes": torch.zeros((0, 4)), "labels": torch.zeros(0, dtype=torch.int64)}

        image = Image.open(img_path).convert("RGB")
        width, height = image.size

        with open(label_path, "r") as f:
            boxes = []
            labels = []
            for line in f.readlines():
                parts = list(map(float, line.strip().split()))
                if len(parts) != 5:
                    continue
                class_id, x, y, w, h = parts
                x_min = (x - w / 2) * width
                y_min = (y - h / 2) * height
                x_max = (x + w / 2) * width
                y_max = (y + h / 2) * height
                boxes.append([x_min, y_min, x_max, y_max])
                labels.append(int(class_id))

        boxes = torch.as_tensor(boxes, dtype=torch.float32)
        labels = torch.as_tensor(labels, dtype=torch.int64)
        target = {"boxes": boxes, "labels": labels}

        if self.transform:
            image = self.transform(image)

        return image, target

# Função para remover itens None do DataLoader
def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    return tuple(zip(*batch)) if batch else ([], [])

# Transformações
transform = transforms.Compose([transforms.ToTensor()])

# Criar DataLoaders
print("📊 Criando DataLoaders...")
train_dataset = ObjectDetectionDataset(train_dir, transform=transform)
val_dataset = ObjectDetectionDataset(val_dir, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False, collate_fn=collate_fn)
print("✅ DataLoaders criados!")

# Definir otimizador
optimizer = torch.optim.SGD(model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005)

# Treinar Modelo
num_epochs = 7
loss_history = []

print("🚀 Iniciando treinamento...")
for epoch in range(num_epochs):
    total_loss = 0
    model.train()

    for batch_idx, (images, targets) in enumerate(train_loader):
        if not images:
            continue

        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]

        optimizer.zero_grad()
        loss_dict = model(images, targets)
        loss = sum(loss for loss in loss_dict.values())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        print(f"🟢 Batch {batch_idx + 1}/{len(train_loader)} - Loss: {loss.item():.4f}")

    loss_history.append(total_loss)
    print(f"✅ Época {epoch + 1}/{num_epochs} - Loss Total: {total_loss:.4f}")

print("⏳ Treinamento finalizado!")

# Salvar Modelo Completo
torch.save(model, "faster_rcnn_vgg16.pt")
print("💾 Modelo salvo como 'faster_rcnn_vgg16.pt'!")

# 📊 Box Plot da Loss ao longo das épocas
plt.figure(figsize=(8, 5))
plt.boxplot(loss_history, vert=True, patch_artist=True)
plt.xlabel('Épocas')
plt.ylabel('Loss Total')
plt.title('Distribuição da Loss ao Longo do Treinamento (VGG16)')
plt.grid()
plt.show()
